In [1]:
# ================================
# 1. IMPORT LIBRARIES
# ================================
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, classification_report,
                             confusion_matrix)
from tqdm import tqdm
import joblib


# ================================
# 2. DATASET PATH
# ================================
dataset_path = "/kaggle/input/datasets/pravallikann/combined-anemia/Fingernails"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)


# ================================
# 3. IMAGE PREPROCESSING FUNCTION
# ================================
def preprocess_image(image_path):
    """
    Reads and preprocesses a single image.
    Returns: numpy array (H, W, 3) uint8 RGB, or None on failure.
    """
    img = cv2.imread(image_path)
    if img is None:
        return None

    # BGR → RGB
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Threshold to remove dark background
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    _, thresh = cv2.threshold(gray, 15, 255, cv2.THRESH_BINARY)

    # Find largest contour (fingernail ROI)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if len(contours) == 0:
        return None

    largest_contour = max(contours, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(largest_contour)
    roi = img_rgb[y:y+h, x:x+w]

    # Resize to ResNet-50 native input size (224x224)
    roi = cv2.resize(roi, (224, 224))

    return roi  # uint8 RGB


# ================================
# 4. PYTORCH DATASET CLASS
# ================================
class AnemiaDataset(Dataset):
    """
    Custom Dataset for anemia fingernail images.
    Applies torchvision transforms (augmentation + normalization).
    """
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels      = labels
        self.transform   = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = preprocess_image(self.image_paths[idx])

        if img is None:
            img = np.zeros((224, 224, 3), dtype=np.uint8)

        from PIL import Image
        img_pil = Image.fromarray(img)

        if self.transform:
            img_tensor = self.transform(img_pil)
        else:
            img_tensor = transforms.ToTensor()(img_pil)

        return img_tensor, self.labels[idx]


# ================================
# 5. TRANSFORMS (TRAIN & VAL)
# ================================
# ImageNet mean/std — standard for ResNet pretrained weights
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(degrees=20),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


# ================================
# 6. LOAD DATASET & BUILD FILE LISTS
# ================================
image_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff'}

def find_class_folders(root_path):
    class_folders = {}
    for dirpath, dirnames, filenames in os.walk(root_path):
        images = [f for f in filenames if os.path.splitext(f)[1].lower() in image_extensions]
        if images:
            folder_name = os.path.basename(dirpath)
            class_folders[folder_name] = dirpath
    return class_folders

found = find_class_folders(dataset_path)
print("Auto-detected class folders:")
for name, path in found.items():
    print(f"  '{name}' → {path}")

image_paths = []
labels      = []

# ⚠️ Update keys below to match your actual folder names
class_to_label = {
    "Anemic":    0,
    "NonAnemic": 1
}
classes = list(class_to_label.keys())

for class_name, label in class_to_label.items():
    if class_name not in found:
        print(f"WARNING: '{class_name}' not found! Available: {list(found.keys())}")
        continue
    class_folder = found[class_name]
    for img_name in tqdm(os.listdir(class_folder), desc=f"Loading {class_name}"):
        img_path = os.path.join(class_folder, img_name)
        image_paths.append(img_path)
        labels.append(label)

image_paths = np.array(image_paths)
labels      = np.array(labels)

print(f"\nTotal images : {len(image_paths)}")
print(f"Class distribution → {classes[0]}: {(labels==0).sum()}, {classes[1]}: {(labels==1).sum()}")


# ================================
# 7. TRAIN-TEST SPLIT (STRATIFIED 80-20)
# ================================
X_train_paths, X_test_paths, y_train, y_test = train_test_split(
    image_paths, labels,
    test_size=0.2,
    stratify=labels,
    random_state=42
)

print(f"Train samples : {len(X_train_paths)}")
print(f"Test  samples : {len(X_test_paths)}")


# ================================
# 8. DATALOADERS
# ================================
BATCH_SIZE = 32   # ResNet-50 is lighter than B4, so 32 is fine

train_dataset = AnemiaDataset(X_train_paths, y_train, transform=train_transform)
test_dataset  = AnemiaDataset(X_test_paths,  y_test,  transform=val_transform)

train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
test_loader   = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)


# ================================
# 9. BUILD RESNET-50 MODEL
# ================================
def build_resnet50(num_classes=2, dropout_rate=0.4):
    """
    Loads pretrained ResNet-50 and replaces the FC head.
    """
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)

    # Freeze all backbone layers
    for param in model.parameters():
        param.requires_grad = False

    # Unfreeze layer3 and layer4 for fine-tuning
    for layer in [model.layer3, model.layer4]:
        for param in layer.parameters():
            param.requires_grad = True

    # Replace fully connected head
    # ResNet-50 fc: Linear(2048 → 1000) by default
    in_features = model.fc.in_features   # 2048
    model.fc = nn.Sequential(
        nn.Dropout(p=dropout_rate),
        nn.Linear(in_features, 512),
        nn.ReLU(),
        nn.BatchNorm1d(512),
        nn.Dropout(p=0.3),
        nn.Linear(512, num_classes)
    )

    return model


model = build_resnet50(num_classes=2, dropout_rate=0.4)
model = model.to(DEVICE)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTrainable parameters: {trainable:,}")


# ================================
# 10. LOSS, OPTIMIZER & SCHEDULER
# ================================
# Weighted loss for class imbalance
class_counts  = np.bincount(y_train)
class_weights = 1.0 / class_counts
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4,
    weight_decay=1e-4
)

# Cosine Annealing LR scheduler
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=25, eta_min=1e-6)


# ================================
# 11. TRAINING LOOP
# ================================
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total   = 0

    for images, labels_batch in tqdm(loader, desc="Training", leave=False):
        images, labels_batch = images.to(device), labels_batch.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels_batch)
        loss.backward()

        # Gradient clipping to prevent exploding gradients
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds         = outputs.argmax(dim=1)
        correct      += (preds == labels_batch).sum().item()
        total        += images.size(0)

    return running_loss / total, correct / total


def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct  = 0
    total    = 0
    all_probs  = []
    all_preds  = []
    all_labels = []

    with torch.no_grad():
        for images, labels_batch in tqdm(loader, desc="Evaluating", leave=False):
            images, labels_batch = images.to(device), labels_batch.to(device)

            outputs = model(images)
            loss    = criterion(outputs, labels_batch)
            probs   = torch.softmax(outputs, dim=1)[:, 1]
            preds   = outputs.argmax(dim=1)

            running_loss += loss.item() * images.size(0)
            correct      += (preds == labels_batch).sum().item()
            total        += images.size(0)

            all_probs.extend(probs.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels_batch.cpu().numpy())

    return (running_loss / total, correct / total,
            np.array(all_preds), np.array(all_probs), np.array(all_labels))


# ================================
# MAIN TRAINING RUN
# ================================
NUM_EPOCHS      = 11
best_val_acc    = 0.0
best_model_path = "/kaggle/working/best_resnet50_anemia.pth"

train_loss_history = []
val_loss_history   = []
train_acc_history  = []
val_acc_history    = []

for epoch in range(1, NUM_EPOCHS + 1):

    train_loss, train_acc          = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
    val_loss, val_acc, _, _, _     = evaluate(model, test_loader, criterion, DEVICE)

    scheduler.step()

    train_loss_history.append(train_loss)
    val_loss_history.append(val_loss)
    train_acc_history.append(train_acc)
    val_acc_history.append(val_acc)

    print(f"Epoch [{epoch:02d}/{NUM_EPOCHS}] "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), best_model_path)
        print(f"  ✅ Best model saved (Val Acc: {best_val_acc:.4f})")


# ================================
# 12. TRAINING CURVES
# ================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(train_loss_history, label="Train Loss", color="blue")
axes[0].plot(val_loss_history,   label="Val Loss",   color="orange")
axes[0].set_title("ResNet-50 — Loss Curve")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()

axes[1].plot(train_acc_history, label="Train Acc", color="green")
axes[1].plot(val_acc_history,   label="Val Acc",   color="red")
axes[1].set_title("ResNet-50 — Accuracy Curve")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()

plt.tight_layout()
plt.savefig("/kaggle/working/resnet50_training_curves.png", dpi=150)
plt.show()


# ================================
# 13. SAVE FINAL MODEL
# ================================
save_path = "/kaggle/working/anemia_resnet50_final.pth"
torch.save(model.state_dict(), save_path)
print(f"\nFinal model saved at: {save_path}")


# ================================
# 14. LOAD BEST MODEL & PREDICT
# ================================
model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
model.eval()
print("Best model loaded successfully!")

_, _, y_pred, y_prob, y_true = evaluate(model, test_loader, criterion, DEVICE)


# ================================
# 15. EVALUATION METRICS
# ================================
accuracy  = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall    = recall_score(y_true, y_pred)
f1        = f1_score(y_true, y_pred)
roc_auc   = roc_auc_score(y_true, y_prob)

print("\n===== MODEL PERFORMANCE =====")
print("Accuracy  :", round(accuracy,  4))
print("Precision :", round(precision, 4))
print("Recall    :", round(recall,    4))
print("F1 Score  :", round(f1,        4))
print("ROC-AUC   :", round(roc_auc,   4))

print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, target_names=classes))


# ================================
# 16. CONFUSION MATRIX
# ================================
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.colorbar(im, ax=ax)
ax.set_xticks([0, 1]); ax.set_xticklabels(classes)
ax.set_yticks([0, 1]); ax.set_yticklabels(classes)
ax.set_xlabel("Predicted Label")
ax.set_ylabel("True Label")
ax.set_title("ResNet-50 — Confusion Matrix")

for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                color='white' if cm[i, j] > cm.max()/2 else 'black', fontsize=14)

plt.tight_layout()
plt.savefig("/kaggle/working/resnet50_confusion_matrix.png", dpi=150)
plt.show()


# ================================
# 17. REAL IMAGE PREDICTION
# ================================
def predict_anemia(image_path, model):
    """
    Predicts Anemic / Non-Anemic for a single fingernail image.
    """
    processed_img = preprocess_image(image_path)
    if processed_img is None:
        print("Image processing failed.")
        return

    features = extract_features(processed_img)
    features = np.array(features).reshape(1, -1)

    prediction   = model.predict(features)[0]
    prob_anemic  = model.predict_proba(features)[0][0]   # index 0 → Anemic
    prob_nonanemic = model.predict_proba(features)[0][1] # index 1 → Non-Anemic

    if prediction == 0:
        result      = "Anemic"
        probability = prob_anemic
    else:
        result      = "Non-Anemic"
        probability = prob_nonanemic

    print("\nPrediction              :", result)
    print(f"Probability ({result}):", round(probability, 4))

In [2]:
# Example usage:
predict_anemia("/kaggle/input/datasets/pravallikann/kkkkkkk/WhatsApp Image 2026-03-03 at 18.45.26.jpeg", model)
predict_anemia("/kaggle/input/datasets/pravallikann/kkkkkkk/WhatsApp Image 2026-03-14 at 12.28.41 (1).jpeg", model)
predict_anemia("/kaggle/input/datasets/pravallikann/kkkkkkk/WhatsApp Image 2026-03-14 at 12.29.07 (1).jpeg", model)

In [3]:
# Example usage:
predict_anemia("/kaggle/input/datasets/pravallikann/kkkkkkk/WhatsApp Image 2026-03-03 at 18.45.26.jpeg", loaded_model)
predict_anemia("/kaggle/input/datasets/pravallikann/kkkkkkk/WhatsApp Image 2026-03-14 at 12.28.41 (1).jpeg", loaded_model)
predict_anemia("/kaggle/input/datasets/pravallikann/kkkkkkk/WhatsApp Image 2026-03-14 at 12.29.07 (1).jpeg", loaded_model)

In [4]:
# Example usage:
image_path = "/kaggle/input/datasets/pravallikann/kkkkkkk/WhatsApp Image 2026-03-03 at 18.45.26.jpeg"   # ← change filename here
predict_anemia(image_path, model)

In [5]:
# ================================
# 17. REAL IMAGE PREDICTION (ResNet-50)
# ================================
def predict_anemia(image_path, model, device=DEVICE):

    processed_img = preprocess_image(image_path)
    if processed_img is None:
        print("Image processing failed.")
        return

    from PIL import Image
    img_pil    = Image.fromarray(processed_img)
    img_tensor = val_transform(img_pil).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        output         = model(img_tensor)
        probs          = torch.softmax(output, dim=1)[0]
        prediction     = output.argmax(dim=1).item()
        prob_anemic    = probs[0].item()
        prob_nonanemic = probs[1].item()

    if prediction == 0:
        result      = "Anemic"
        probability = prob_anemic
    else:
        result      = "Non-Anemic"
        probability = prob_nonanemic

    print("\nPrediction              :", result)
    print(f"Probability ({result}) :", round(probability, 4))


# ================================
# PREDICT ON REAL IMAGE
# ================================

# Step 1: Upload your fingernail image via Kaggle Upload button
# Step 2: Run this cell with your image filename

image_path = "/kaggle/working/your_image.jpg"   # ← change filename here
predict_anemia(image_path, model)

In [6]:
# ================================
# 17. REAL IMAGE PREDICTION (ResNet-50)
# ================================
def predict_anemia(image_path, model, device=DEVICE):

    processed_img = preprocess_image(image_path)
    if processed_img is None:
        print("Image processing failed.")
        return

    from PIL import Image
    img_pil    = Image.fromarray(processed_img)
    img_tensor = val_transform(img_pil).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        output         = model(img_tensor)
        probs          = torch.softmax(output, dim=1)[0]
        prediction     = output.argmax(dim=1).item()
        prob_anemic    = probs[0].item()
        prob_nonanemic = probs[1].item()

    if prediction == 0:
        result      = "Anemic"
        probability = prob_anemic
    else:
        result      = "Non-Anemic"
        probability = prob_nonanemic

    print("\nPrediction              :", result)
    print(f"Probability ({result}) :", round(probability, 4))


# ================================
# PREDICT ON REAL IMAGE
# ================================

# Step 1: Upload your fingernail image via Kaggle Upload button
# Step 2: Run this cell with your image filename

image_path = "/kaggle/input/datasets/pravallikann/kkkkkkk/WhatsApp Image 2026-03-03 at 18.45.26.jpeg"   # ← change filename here
predict_anemia(image_path, model)

In [7]:
# ================================
# 18. SAVE NOTEBOOK
# ================================
import subprocess

result = subprocess.run(
    ["jupyter", "nbconvert", "--to", "notebook",
     "--output", "/kaggle/working/anemia_resnet50_notebook.ipynb",
     "/kaggle/input/__notebook_source__/notebook.ipynb"],
    capture_output=True, text=True
)

if result.returncode == 0:
    print("✅ Notebook saved at: /kaggle/working/anemia_resnet50_notebook.ipynb")
else:
    print("❌ Error:", result.stderr)

In [8]:
import glob, shutil

notebooks = glob.glob("/kaggle/**/*.ipynb", recursive=True)
notebooks = [n for n in notebooks if "working" not in n]
print("Found:", notebooks)

if notebooks:
    shutil.copy(notebooks[0], "/kaggle/working/anemia_resnet50_notebook.ipynb")
    print("✅ Notebook saved at: /kaggle/working/anemia_resnet50_notebook.ipynb")

In [9]:
# ================================
# SAVE NOTEBOOK — KAGGLE METHOD
# ================================
import json
import os

# Method 1: using ipynb kernel connection file
try:
    import ipykernel
    kernel_id = ipykernel.connect.get_connection_file()
    print("Kernel file:", kernel_id)
except:
    pass


# Method 2: Direct Kaggle notebook source path
import shutil, glob

# Try all known Kaggle notebook locations
possible_paths = [
    "/kaggle/input/__notebook_source__/notebook.ipynb",
    "/kaggle/input/__notebook_source__/*.ipynb",
    "/tmp/*.ipynb",
    "/tmp/**/*.ipynb",
    "/root/*.ipynb",
    "/root/**/*.ipynb",
]

found = []
for path in possible_paths:
    matches = glob.glob(path, recursive=True)
    found.extend(matches)

print("Found notebooks:", found)

if found:
    shutil.copy(found[0], "/kaggle/working/anemia_resnet50_notebook.ipynb")
    print("✅ Saved at: /kaggle/working/anemia_resnet50_notebook.ipynb")

else:
    # Method 3: nbconvert from running kernel
    import subprocess
    result = subprocess.run(
        ["find", "/", "-name", "*.ipynb", "-not", "-path", "*/working/*"],
        capture_output=True, text=True, timeout=10
    )
    print("Find output:\n", result.stdout)

In [10]:
# ================================
# SAVE NOTEBOOK — KAGGLE METHOD
# ================================
import json
import os

# Method 1: using ipynb kernel connection file
try:
    import ipykernel
    kernel_id = ipykernel.connect.get_connection_file()
    print("Kernel file:", kernel_id)
except:
    pass


# Method 2: Direct Kaggle notebook source path
import shutil, glob

# Try all known Kaggle notebook locations
possible_paths = [
    "/kaggle/input/__notebook_source__/notebook.ipynb",
    "/kaggle/input/__notebook_source__/*.ipynb",
    "/tmp/*.ipynb",
    "/tmp/**/*.ipynb",
    "/root/*.ipynb",
    "/root/**/*.ipynb",
]

found = []
for path in possible_paths:
    matches = glob.glob(path, recursive=True)
    found.extend(matches)

print("Found notebooks:", found)

if found:
    shutil.copy(found[0], "/kaggle/working/anemia_resnet50_notebook.ipynb")
    print("✅ Saved at: /kaggle/working/anemia_resnet50_notebook.ipynb")

else:
    # Method 3: nbconvert from running kernel
    import subprocess
    result = subprocess.run(
        ["find", "/", "-name", "*.ipynb", "-not", "-path", "*/working/*"],
        capture_output=True, text=True, timeout=10
    )
    print("Find output:\n", result.stdout)

In [11]:
# ================================
# SAVE NOTEBOOK VIA KERNEL CONNECTION
# ================================
import json, shutil, os, glob

# Step 1: Read kernel connection file to get kernel ID
kernel_file = "/root/.local/share/jupyter/runtime/kernel-cdad30bf-ac0b-40de-9733-57841d8caaeb.json"

with open(kernel_file, 'r') as f:
    kernel_info = json.load(f)
print("Kernel info:", kernel_info)

# Step 2: Search only specific directories (avoid full system scan)
search_dirs = [
    "/root/.local/share/jupyter/",
    "/root/",
    "/tmp/",
    "/kaggle/",
]

found = []
for d in search_dirs:
    if os.path.exists(d):
        matches = glob.glob(os.path.join(d, "*.ipynb"))
        found.extend(matches)
        print(f"Searched {d} → {matches}")

print("\nFound notebooks:", found)

if found:
    shutil.copy(found[0], "/kaggle/working/anemia_resnet50_notebook.ipynb")
    print("✅ Saved at: /kaggle/working/anemia_resnet50_notebook.ipynb")

In [12]:
# ================================
# SAVE NOTEBOOK VIA KERNEL CONNECTION
# ================================
import json, shutil, os, glob

# Step 1: Read kernel connection file to get kernel ID
kernel_file = "/root/.local/share/jupyter/runtime/kernel-cdad30bf-ac0b-40de-9733-57841d8caaeb.json"

with open(kernel_file, 'r') as f:
    kernel_info = json.load(f)
print("Kernel info:", kernel_info)

# Step 2: Search only specific directories (avoid full system scan)
search_dirs = [
    "/root/.local/share/jupyter/",
    "/root/",
    "/tmp/",
    "/kaggle/",
]

found = []
for d in search_dirs:
    if os.path.exists(d):
        matches = glob.glob(os.path.join(d, "*.ipynb"))
        found.extend(matches)
        print(f"Searched {d} → {matches}")

print("\nFound notebooks:", found)

if found:
    shutil.copy(found[0], "/kaggle/working/anemia_resnet50_notebook.ipynb")
    print("✅ Saved at: /kaggle/working/anemia_resnet50_notebook.ipynb")

In [13]:
# ================================
# SAVE NOTEBOOK — BUILD .ipynb FROM
# SESSION HISTORY
# ================================
import IPython, json

ip   = IPython.get_ipython()
hist = list(ip.history_manager.get_range(output=False))

# Build proper .ipynb structure
notebook = {
    "nbformat": 4,
    "nbformat_minor": 5,
    "metadata": {
        "kernelspec": {
            "display_name": "Python 3",
            "language": "python",
            "name": "python3"
        },
        "language_info": {
            "name": "python",
            "version": "3.12.0"
        }
    },
    "cells": []
}

for session, line_num, src in hist:
    if src.strip() == "":
        continue
    cell = {
        "cell_type": "code",
        "execution_count": line_num,
        "metadata": {},
        "outputs": [],
        "source": src
    }
    notebook["cells"].append(cell)

# Save as .ipynb
save_path = "/kaggle/working/anemia_resnet50_notebook.ipynb"
with open(save_path, "w", encoding="utf-8") as f:
    json.dump(notebook, f, indent=2, ensure_ascii=False)

print(f"✅ Notebook saved at: {save_path}")
print(f"   Total cells : {len(notebook['cells'])}")
print(f"   File size   : {round(os.path.getsize(save_path)/1024, 2)} KB")
print("\n👉 Go to Output tab → Download anemia_resnet50_notebook.ipynb")

In [14]:
# ================================
# SAVE NOTEBOOK — BUILD .ipynb FROM
# SESSION HISTORY
# ================================
import IPython, json

ip   = IPython.get_ipython()
hist = list(ip.history_manager.get_range(output=False))

# Build proper .ipynb structure
notebook = {
    "nbformat": 4,
    "nbformat_minor": 5,
    "metadata": {
        "kernelspec": {
            "display_name": "Python 3",
            "language": "python",
            "name": "python3"
        },
        "language_info": {
            "name": "python",
            "version": "3.12.0"
        }
    },
    "cells": []
}

for session, line_num, src in hist:
    if src.strip() == "":
        continue
    cell = {
        "cell_type": "code",
        "execution_count": line_num,
        "metadata": {},
        "outputs": [],
        "source": src
    }
    notebook["cells"].append(cell)

# Save as .ipynb
save_path = "/kaggle/working/anemia_resnet50_notebook.ipynb"
with open(save_path, "w", encoding="utf-8") as f:
    json.dump(notebook, f, indent=2, ensure_ascii=False)

print(f"✅ Notebook saved at: {save_path}")
print(f"   Total cells : {len(notebook['cells'])}")
print(f"   File size   : {round(os.path.getsize(save_path)/1024, 2)} KB")
print("\n👉 Go to Output tab → Download anemia_resnet50_notebook.ipynb")

In [15]:
# ================================
# 1. IMPORT LIBRARIES
# ================================
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, classification_report,
                             confusion_matrix)
from tqdm import tqdm
import joblib


# ================================
# 2. DATASET PATH
# ================================
dataset_path = "/kaggle/input/datasets/pravallikann/combined-anemia/Fingernails"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)


# ================================
# 3. IMAGE PREPROCESSING FUNCTION
# ================================
def preprocess_image(image_path):
    """
    Reads and preprocesses a single image.
    Returns: numpy array (H, W, 3) uint8 RGB, or None on failure.
    """
    img = cv2.imread(image_path)
    if img is None:
        return None

    # BGR → RGB
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Threshold to remove dark background
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    _, thresh = cv2.threshold(gray, 15, 255, cv2.THRESH_BINARY)

    # Find largest contour (fingernail ROI)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if len(contours) == 0:
        return None

    largest_contour = max(contours, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(largest_contour)
    roi = img_rgb[y:y+h, x:x+w]

    # Resize to ResNet-50 native input size (224x224)
    roi = cv2.resize(roi, (224, 224))

    return roi  # uint8 RGB


# ================================
# 4. PYTORCH DATASET CLASS
# ================================
class AnemiaDataset(Dataset):
    """
    Custom Dataset for anemia fingernail images.
    Applies torchvision transforms (augmentation + normalization).
    """
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels      = labels
        self.transform   = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = preprocess_image(self.image_paths[idx])

        if img is None:
            img = np.zeros((224, 224, 3), dtype=np.uint8)

        from PIL import Image
        img_pil = Image.fromarray(img)

        if self.transform:
            img_tensor = self.transform(img_pil)
        else:
            img_tensor = transforms.ToTensor()(img_pil)

        return img_tensor, self.labels[idx]


# ================================
# 5. TRANSFORMS (TRAIN & VAL)
# ================================
# ImageNet mean/std — standard for ResNet pretrained weights
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(degrees=20),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


# ================================
# 6. LOAD DATASET & BUILD FILE LISTS
# ================================
image_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff'}

def find_class_folders(root_path):
    class_folders = {}
    for dirpath, dirnames, filenames in os.walk(root_path):
        images = [f for f in filenames if os.path.splitext(f)[1].lower() in image_extensions]
        if images:
            folder_name = os.path.basename(dirpath)
            class_folders[folder_name] = dirpath
    return class_folders

found = find_class_folders(dataset_path)
print("Auto-detected class folders:")
for name, path in found.items():
    print(f"  '{name}' → {path}")

image_paths = []
labels      = []

# ⚠️ Update keys below to match your actual folder names
class_to_label = {
    "Anemic":    0,
    "NonAnemic": 1
}
classes = list(class_to_label.keys())

for class_name, label in class_to_label.items():
    if class_name not in found:
        print(f"WARNING: '{class_name}' not found! Available: {list(found.keys())}")
        continue
    class_folder = found[class_name]
    for img_name in tqdm(os.listdir(class_folder), desc=f"Loading {class_name}"):
        img_path = os.path.join(class_folder, img_name)
        image_paths.append(img_path)
        labels.append(label)

image_paths = np.array(image_paths)
labels      = np.array(labels)

print(f"\nTotal images : {len(image_paths)}")
print(f"Class distribution → {classes[0]}: {(labels==0).sum()}, {classes[1]}: {(labels==1).sum()}")


# ================================
# 7. TRAIN-TEST SPLIT (STRATIFIED 80-20)
# ================================
X_train_paths, X_test_paths, y_train, y_test = train_test_split(
    image_paths, labels,
    test_size=0.2,
    stratify=labels,
    random_state=42
)

print(f"Train samples : {len(X_train_paths)}")
print(f"Test  samples : {len(X_test_paths)}")


# ================================
# 8. DATALOADERS
# ================================
BATCH_SIZE = 32   # ResNet-50 is lighter than B4, so 32 is fine

train_dataset = AnemiaDataset(X_train_paths, y_train, transform=train_transform)
test_dataset  = AnemiaDataset(X_test_paths,  y_test,  transform=val_transform)

train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
test_loader   = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)


# ================================
# 9. BUILD RESNET-50 MODEL
# ================================
def build_resnet50(num_classes=2, dropout_rate=0.4):
    """
    Loads pretrained ResNet-50 and replaces the FC head.
    """
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)

    # Freeze all backbone layers
    for param in model.parameters():
        param.requires_grad = False

    # Unfreeze layer3 and layer4 for fine-tuning
    for layer in [model.layer3, model.layer4]:
        for param in layer.parameters():
            param.requires_grad = True

    # Replace fully connected head
    # ResNet-50 fc: Linear(2048 → 1000) by default
    in_features = model.fc.in_features   # 2048
    model.fc = nn.Sequential(
        nn.Dropout(p=dropout_rate),
        nn.Linear(in_features, 512),
        nn.ReLU(),
        nn.BatchNorm1d(512),
        nn.Dropout(p=0.3),
        nn.Linear(512, num_classes)
    )

    return model


model = build_resnet50(num_classes=2, dropout_rate=0.4)
model = model.to(DEVICE)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTrainable parameters: {trainable:,}")


# ================================
# 10. LOSS, OPTIMIZER & SCHEDULER
# ================================
# Weighted loss for class imbalance
class_counts  = np.bincount(y_train)
class_weights = 1.0 / class_counts
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4,
    weight_decay=1e-4
)

# Cosine Annealing LR scheduler
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=25, eta_min=1e-6)


# ================================
# 11. TRAINING LOOP
# ================================
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total   = 0

    for images, labels_batch in tqdm(loader, desc="Training", leave=False):
        images, labels_batch = images.to(device), labels_batch.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels_batch)
        loss.backward()

        # Gradient clipping to prevent exploding gradients
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds         = outputs.argmax(dim=1)
        correct      += (preds == labels_batch).sum().item()
        total        += images.size(0)

    return running_loss / total, correct / total


def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct  = 0
    total    = 0
    all_probs  = []
    all_preds  = []
    all_labels = []

    with torch.no_grad():
        for images, labels_batch in tqdm(loader, desc="Evaluating", leave=False):
            images, labels_batch = images.to(device), labels_batch.to(device)

            outputs = model(images)
            loss    = criterion(outputs, labels_batch)
            probs   = torch.softmax(outputs, dim=1)[:, 1]
            preds   = outputs.argmax(dim=1)

            running_loss += loss.item() * images.size(0)
            correct      += (preds == labels_batch).sum().item()
            total        += images.size(0)

            all_probs.extend(probs.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels_batch.cpu().numpy())

    return (running_loss / total, correct / total,
            np.array(all_preds), np.array(all_probs), np.array(all_labels))


# ================================
# MAIN TRAINING RUN
# ================================
NUM_EPOCHS      = 11
best_val_acc    = 0.0
best_model_path = "/kaggle/working/best_resnet50_anemia.pth"

train_loss_history = []
val_loss_history   = []
train_acc_history  = []
val_acc_history    = []

for epoch in range(1, NUM_EPOCHS + 1):

    train_loss, train_acc          = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
    val_loss, val_acc, _, _, _     = evaluate(model, test_loader, criterion, DEVICE)

    scheduler.step()

    train_loss_history.append(train_loss)
    val_loss_history.append(val_loss)
    train_acc_history.append(train_acc)
    val_acc_history.append(val_acc)

    print(f"Epoch [{epoch:02d}/{NUM_EPOCHS}] "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), best_model_path)
        print(f"  ✅ Best model saved (Val Acc: {best_val_acc:.4f})")


# ================================
# 12. TRAINING CURVES
# ================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(train_loss_history, label="Train Loss", color="blue")
axes[0].plot(val_loss_history,   label="Val Loss",   color="orange")
axes[0].set_title("ResNet-50 — Loss Curve")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()

axes[1].plot(train_acc_history, label="Train Acc", color="green")
axes[1].plot(val_acc_history,   label="Val Acc",   color="red")
axes[1].set_title("ResNet-50 — Accuracy Curve")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()

plt.tight_layout()
plt.savefig("/kaggle/working/resnet50_training_curves.png", dpi=150)
plt.show()


# ================================
# 13. SAVE FINAL MODEL
# ================================
save_path = "/kaggle/working/anemia_resnet50_final.pth"
torch.save(model.state_dict(), save_path)
print(f"\nFinal model saved at: {save_path}")


# ================================
# 14. LOAD BEST MODEL & PREDICT
# ================================
model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
model.eval()
print("Best model loaded successfully!")

_, _, y_pred, y_prob, y_true = evaluate(model, test_loader, criterion, DEVICE)


# ================================
# 15. EVALUATION METRICS
# ================================
accuracy  = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall    = recall_score(y_true, y_pred)
f1        = f1_score(y_true, y_pred)
roc_auc   = roc_auc_score(y_true, y_prob)

print("\n===== MODEL PERFORMANCE =====")
print("Accuracy  :", round(accuracy,  4))
print("Precision :", round(precision, 4))
print("Recall    :", round(recall,    4))
print("F1 Score  :", round(f1,        4))
print("ROC-AUC   :", round(roc_auc,   4))

print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, target_names=classes))


# ================================
# 16. CONFUSION MATRIX
# ================================
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.colorbar(im, ax=ax)
ax.set_xticks([0, 1]); ax.set_xticklabels(classes)
ax.set_yticks([0, 1]); ax.set_yticklabels(classes)
ax.set_xlabel("Predicted Label")
ax.set_ylabel("True Label")
ax.set_title("ResNet-50 — Confusion Matrix")

for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                color='white' if cm[i, j] > cm.max()/2 else 'black', fontsize=14)

plt.tight_layout()
plt.savefig("/kaggle/working/resnet50_confusion_matrix.png", dpi=150)
plt.show()

In [16]:
# ================================
# 17. REAL IMAGE PREDICTION (ResNet-50)
# ================================
def predict_anemia(image_path, model, device=DEVICE):

    processed_img = preprocess_image(image_path)
    if processed_img is None:
        print("Image processing failed.")
        return

    from PIL import Image
    img_pil    = Image.fromarray(processed_img)
    img_tensor = val_transform(img_pil).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        output         = model(img_tensor)
        probs          = torch.softmax(output, dim=1)[0]
        prediction     = output.argmax(dim=1).item()
        prob_anemic    = probs[0].item()
        prob_nonanemic = probs[1].item()

    if prediction == 0:
        result      = "Anemic"
        probability = prob_anemic
    else:
        result      = "Non-Anemic"
        probability = prob_nonanemic

    print("\nPrediction              :", result)
    print(f"Probability ({result}) :", round(probability, 4))


# ================================
# PREDICT ON REAL IMAGE
# ================================

# Step 1: Upload your fingernail image via Kaggle Upload button
# Step 2: Run this cell with your image filename

image_path = "/kaggle/input/datasets/pravallikann/kkkkkkk/WhatsApp Image 2026-03-03 at 18.45.26.jpeg"   # ← change filename here
predict_anemia(image_path, model)

In [17]:
# ================================
# SAVE NOTEBOOK — BUILD .ipynb FROM
# SESSION HISTORY
# ================================
import IPython, json

ip   = IPython.get_ipython()
hist = list(ip.history_manager.get_range(output=False))

# Build proper .ipynb structure
notebook = {
    "nbformat": 4,
    "nbformat_minor": 5,
    "metadata": {
        "kernelspec": {
            "display_name": "Python 3",
            "language": "python",
            "name": "python3"
        },
        "language_info": {
            "name": "python",
            "version": "3.12.0"
        }
    },
    "cells": []
}

for session, line_num, src in hist:
    if src.strip() == "":
        continue
    cell = {
        "cell_type": "code",
        "execution_count": line_num,
        "metadata": {},
        "outputs": [],
        "source": src
    }
    notebook["cells"].append(cell)

# Save as .ipynb
save_path = "/kaggle/working/anemia_resnet50_notebook.ipynb"
with open(save_path, "w", encoding="utf-8") as f:
    json.dump(notebook, f, indent=2, ensure_ascii=False)

print(f"✅ Notebook saved at: {save_path}")
print(f"   Total cells : {len(notebook['cells'])}")
print(f"   File size   : {round(os.path.getsize(save_path)/1024, 2)} KB")
print("\n👉 Go to Output tab → Download anemia_resnet50_notebook.ipynb")